# 04 - Huấn Luyện Mô Hình Physics-Informed (ThoR PINN)
### Dự án: Dự Báo Lượng Mưa Cực Ngắn (Rainfall Nowcasting) tại Việt Nam
---

Mô hình kết hợp giữa mạng học sâu Spatiotemporal và các định luật vật lý khí quyển:
1. **Lớp Ràng buộc Cứng TFC (Theory of Functional Connections):** Ép buộc điều kiện ban đầu $t=0$ chính xác tuyệt đối:
   $$\hat{P}(t) = P_0 + \mathbf{N}(t) \cdot (1 - e^{-t})$$
2. **Hàm Phạt Vật Lý (Advection-Diffusion PDE Loss):** Phạt các dự báo mâu thuẫn với trường vector gió $(u_{10}, v_{10})$:
   $$\mathcal{R}_{\text{PDE}} = \frac{\partial P}{\partial t} + u_{10} \frac{\partial P}{\partial x} + v_{10} \frac{\partial P}{\partial y} - D \nabla^2 P$$


In [ ]:
import os
import json
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from dataset import load_processed_loaders

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Thiết bị tính toán đang sử dụng: {device}")


## 1. Nạp Dữ liệu Đa biến


In [ ]:
train_loader, val_loader, test_loader, metadata = load_processed_loaders(
    processed_dir="processed_data",
    batch_size=8,
    in_steps=6,
    out_steps=6,
    filter_dry_samples=True
)
print("Đã tải xong DataLoaders.")


## 2. Định nghĩa Hàm Mất Mát Vật Lý (Physics-Informed Advection Loss)


In [ ]:
class PhysicsAdvectionLoss(nn.Module):
    def __init__(self):
        super(PhysicsAdvectionLoss, self).__init__()
        # Bộ lọc Sobel tính đạo hàm không gian dP/dx và dP/dy
        sobel_x = torch.tensor([[-1., 0., 1.],
                                [-2., 0., 2.],
                                [-1., 0., 1.]]).view(1, 1, 3, 3)
        sobel_y = torch.tensor([[-1., -2., -1.],
                                [ 0.,  0.,  0.],
                                [ 1.,  2.,  1.]]).view(1, 1, 3, 3)
        self.register_buffer('sobel_x', sobel_x)
        self.register_buffer('sobel_y', sobel_y)

    def forward(self, P_pred, u_wind, v_wind):
        # P_pred: (B, T, 1, H, W)
        # u_wind, v_wind: (B, T, 1, H, W) hoặc mốc gió t=0
        B, T, C, H, W = P_pred.shape
        pde_loss = 0.0
        
        for t in range(1, T):
            dp_dt = P_pred[:, t] - P_pred[:, t-1]
            dp_dx = F.conv2d(P_pred[:, t], self.sobel_x, padding=1)
            dp_dy = F.conv2d(P_pred[:, t], self.sobel_y, padding=1)
            
            # Thành phần bình lưu thực tế: u * dP/dx + v * dP/dy
            advection = u_wind * dp_dx + v_wind * dp_dy
            residual = dp_dt + 0.1 * advection
            pde_loss += torch.mean(residual ** 2)
            
        return pde_loss / max(T - 1, 1)


## 3. Định nghĩa Mô hình ThoR-PINN (TFC Constrained NowcastNet)


In [ ]:
class ConvLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size=(3, 3), bias=True):
        super(ConvLSTMCell, self).__init__()
        self.hidden_dim = hidden_dim
        padding = kernel_size[0] // 2, kernel_size[1] // 2
        self.conv = nn.Conv2d(
            in_channels=input_dim + hidden_dim,
            out_channels=4 * hidden_dim,
            kernel_size=kernel_size,
            padding=padding,
            bias=bias
        )

    def forward(self, input_tensor, cur_state):
        h_cur, c_cur = cur_state
        combined = torch.cat([input_tensor, h_cur], dim=1)
        combined_conv = self.conv(combined)
        cc_i, cc_f, cc_o, cc_g = torch.split(combined_conv, self.hidden_dim, dim=1)
        
        i = torch.sigmoid(cc_i)
        f = torch.sigmoid(cc_f)
        o = torch.sigmoid(cc_o)
        g = torch.tanh(cc_g)
        
        c_next = f * c_cur + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, c_next

class ThoR_PINN_Nowcast(nn.Module):
    def __init__(self, in_channels=5, out_channels=1, hidden_dim=32, out_steps=6):
        super(ThoR_PINN_Nowcast, self).__init__()
        self.out_steps = out_steps
        self.hidden_dim = hidden_dim
        
        self.encoder = ConvLSTMCell(in_channels, hidden_dim)
        self.decoder = ConvLSTMCell(hidden_dim, hidden_dim)
        self.out_conv = nn.Sequential(
            nn.Conv2d(hidden_dim, 16, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(16, out_channels, kernel_size=1)
        )

    def forward(self, x):
        b, seq_len, _, h, w = x.size()
        h_enc = torch.zeros(b, self.hidden_dim, h, w, device=x.device)
        c_enc = torch.zeros(b, self.hidden_dim, h, w, device=x.device)
        
        for t in range(seq_len):
            h_enc, c_enc = self.encoder(x[:, t], (h_enc, c_enc))
            
        # P_0 là ảnh lượng mưa thời điểm cuối cùng của quá khứ (t=0)
        P_0 = x[:, -1, 0:1, :, :]
        
        h_dec, c_dec = h_enc, c_enc
        outputs = []
        for t in range(self.out_steps):
            h_dec, c_dec = self.decoder(h_dec, (h_dec, c_dec))
            N_t = self.out_conv(h_dec)
            
            # Căng hàm TFC (Theory of Functional Connections):
            # Y_pred = P_0 + N_t * (1 - e^(-(t+1)))
            time_decay = 1.0 - np.exp(-(t + 1))
            y_pred_t = P_0 + N_t * time_decay
            outputs.append(torch.relu(y_pred_t))
            
        return torch.stack(outputs, dim=1)

model = ThoR_PINN_Nowcast(in_channels=5, out_channels=1, hidden_dim=32, out_steps=6).to(device)
print("Đã khởi tạo mô hình ThoR PINN!")


## 4. Huấn luyện Mô hình ThoR-PINN


In [ ]:
mse_criterion = nn.MSELoss()
physics_criterion = PhysicsAdvectionLoss().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

LAMBDA_PDE = 0.05
EPOCHS = 10
best_pinn_loss = float('inf')

print("Bắt đầu huấn luyện ThoR PINN...")
for epoch in range(EPOCHS):
    model.train()
    epoch_mse = 0.0
    epoch_pde = 0.0
    
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        
        preds = model(batch_x)
        loss_data = mse_criterion(preds, batch_y)
        
        # Trích xuất trường gió u10 (kênh 3), v10 (kênh 4) tại t=0
        u_field = batch_x[:, -1, 3:4, :, :]
        v_field = batch_x[:, -1, 4:5, :, :]
        loss_pde = physics_criterion(preds, u_field, v_field)
        
        total_loss = loss_data + LAMBDA_PDE * loss_pde
        total_loss.backward()
        optimizer.step()
        
        epoch_mse += loss_data.item()
        epoch_pde += loss_pde.item()
        
    epoch_mse /= len(train_loader)
    epoch_pde /= len(train_loader)
    print(f"Epoch [{epoch+1:02d}/{EPOCHS:02d}] - Data MSE: {epoch_mse:.6f} | Physics PDE: {epoch_pde:.6f}")

torch.save(model.state_dict(), 'pinn_best_model.pth')
print("Đã lưu mô hình pinn_best_model.pth thành công!")
